In [ ]:
# Colab Cell 1 — Install dependencies
!pip install --upgrade pip >/dev/null
!pip install azure-storage-blob mysql-connector-python requests tqdm python-dotenv Pillow >/dev/null
!pip install openai

In [ ]:
# Colab Cell 2 — MySQL connection and table setup
import os
from datetime import datetime
import mysql.connector
from mysql.connector import errorcode

MYSQL_HOST = os.getenv("MYSQL_HOST", "cycleai.mysql.database.azure.com")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER", "admindev")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "Zana81333")
MYSQL_DB = os.getenv("MYSQL_DB", "cycle_ai")

def get_mysql_conn():
    cfg = {
        "host": MYSQL_HOST,
        "port": MYSQL_PORT,
        "user": MYSQL_USER,
        "password": MYSQL_PASSWORD,
        "database": MYSQL_DB,
        "autocommit": False,
    }
    try:
        conn = mysql.connector.connect(**cfg)
        return conn
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
            raise

print("MySQL connected and table ensured: perception_poll_votes_ai")


In [ ]:
# Colab Cell 3 — Azure Blob connection and caching blob names
import os, io, random
from azure.storage.blob import BlobServiceClient
from PIL import Image

AZURE_CONN_STR = os.getenv("AZURE_CONN_STR", "")   # recommended: store in Colab secret or env
BLOB_CONTAINER = os.getenv("BLOB_CONTAINER", "london")

if not AZURE_CONN_STR:
    raise RuntimeError("AZURE_CONN_STR not set. Set as environment variable or inline in this cell.")

blob_svc = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
container_client = blob_svc.get_container_client(BLOB_CONTAINER)

print("Listing blobs in 'Images/' folder (this may take a few seconds)...")

IMAGE_PREFIX = "training/training/Images/"
all_image_blobs = container_client.list_blobs(name_starts_with=IMAGE_PREFIX)

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

blob_names = [
    blob.name for blob in all_image_blobs
    if blob.name.lower().endswith(IMAGE_EXTENSIONS)
]

print(f"Found {len(blob_names)} image files.")
if len(blob_names) < 2:
    raise RuntimeError(f"Need at least 2 blobs in container to sample pairs. Found: {len(blob_names)}")

def download_blob_to_pil(blob_name, timeout=30):
    """Download a blob and return it as a PIL Image."""
    blob_data = container_client.download_blob(blob_name, timeout=timeout).readall()
    img = Image.open(io.BytesIO(blob_data)).convert("RGB")
    return img

print("Azure blob helpers ready.")


In [ ]:
# Colab Cell 4 (Modified for GPT-4o + Database Source + Comparison)
import os, io, json, time, base64, hashlib, random, re
from tqdm import tqdm
from PIL import Image
import requests
import mysql.connector
from datetime import datetime
from time import sleep
import uuid
from IPython.display import display, HTML
import pandas as pd
from openai import AzureOpenAI
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ---------- CONFIG ----------
# ---------- AZURE OPENAI CONFIG (minimal changes) ----------

OPENAI_MODEL_ID = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
AZURE_OPENAI_ENDPOINT = os.getenv("DEPLOYMENT_NAME","https://YOUR_AZURE_OPENAI_RESOURCE.openai.azure.com/")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "YOUR_AZURE_OPENAI_API_KEY")

# Construct the client for Azure
client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://YOUR_AZURE_OPENAI_RESOURCE.openai.azure.com/",
    api_key=AZURE_OPENAI_API_KEY,
)


WORKER_ID = os.getenv("WORKER_ID", str(uuid.uuid4()))

N=20000
BATCH_COMMIT = 1
RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.1"))
RETRY_MAX = 3
IMAGE_PREFIX = "training/training/Images/" # Prefix to add to DB filenames

FULL_PROMPT_TEXT = (
    "You will see two images: Image A (first) and Image B (second).\n\n"
    "TASK: Decide which image looks SAFER for cyclists.\n\n"
    "Respond with ONE line only in the exact required format:\n"
    "[VOTE]: [Vote Number] - [reason]\n\n"
    "Where Vote Number is:\n"
    "  0 - Both look equally safe\n"
    "  1 - Image A looks safer\n"
    "  2 - Image B looks safer\n\n"
    "Keep your reason concise. DO NOT include any extra text.\n\n"
    "Examples (Use this format EXACTLY):\n"
    "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
    "[VOTE]: 2 - Image B has fewer vehicles present.\n"
    "[VOTE]: 0 - Both show similar traffic conditions.\n"
    "\nNow decide which image looks safer."
)

# Stats tracking
results = []
agreement_stats = {"matches": 0, "total": 0}

def pil_to_data_uri(pil_img, quality=85):
    b64 = pil_to_base64_jpeg(pil_img, quality=quality)
    return f"data:image/jpeg;base64,{b64}"

def display_pair_url(image_1, image_2, ai_vote, human_vote, reason, name1, name2, max_width=320):
    uri1 = "https://cycleai.net/london/"+image_1
    uri2 = "https://cycleai.net/london/"+image_2

    # Color code match status
    match_color = "green"

    caption = (
        f"<b>AI Vote:</b> {ai_vote}<br>"
        f"<b>Reason:</b> {reason}<br>"
        f"<b>Files:</b> {name1} — {name2}"
    )

    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>A: {name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>B: {name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

def display_pair(pil1, pil2, ai_vote, human_vote, reason, name1, name2, max_width=320):
    uri1 = pil_to_data_uri(pil1)
    uri2 = pil_to_data_uri(pil2)

    # Color code match status
    match_color = "green"

    caption = (
        f"<b>AI Vote:</b> {ai_vote}<br>"
        f"<b>Reason:</b> {reason}<br>"
        f"<b>Files:</b> {name1} — {name2}"
    )

    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>A: {name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>B: {name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

# ---------- UTILITIES ----------
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format="JPEG", quality=quality)
    return base64.b64encode(buff.getvalue()).decode("utf-8")

def idempotency_key(name1, name2):
    s = f"{name1}|{name2}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def upsert_vote(conn, rec):
    cur = conn.cursor()
    sql = """
    INSERT INTO perception_poll_votes_ai
      (image_name1, image_name2, vote, model_id, user_id, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec.get("reason"))
    cur.execute(sql, params)
    cur.close()

def get_source_votes(conn, limit):
    rows = []
    for _ in range(limit):
        img1, img2 = random.sample(blob_names, 2)
        row = {
            "image_name1": img1,
            "image_name2": img2,
        }

        rows.append(row)

    return rows

def call_model_with_public_urls(image_name1, image_name2, prompt_text, image_angle1="", image_angle2=""):
    try:
        uri1 = "https://cycleai.net/london/"+image_name1
        uri2 = "https://cycleai.net/london/"+image_name2

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_completion_tokens=1064
        )

        return response.choices[0].message.content
    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise




def call_model_openai(prompt_text, pil1, pil2):
    try:
        uri1 = pil_to_data_uri(pil1)
        uri2 = pil_to_data_uri(pil2)

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_completion_tokens=1064
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise

print(f"Starting AI-only inference loop. Target N={N}, Model={OPENAI_MODEL_ID}")
conn = get_mysql_conn()
insert_count = 0

# SHOW ENV + basic config that commonly causes mistakes
print("CONFIG CHECK:")
print("  IMAGE_PREFIX =", repr(IMAGE_PREFIX))
print("  AZURE_BLOB_CONTAINER =", repr(os.environ.get("AZURE_BLOB_CONTAINER")))
print("  AZURE_CONN_STR =", "SET" if os.environ.get("AZURE_CONN_STR") else "MISSING")
print("  AZURE_OPENAI_ENDPOINT =", repr(os.environ.get("AZURE_OPENAI_ENDPOINT")))
print("  AZURE_OPENAI_DEPLOYMENT =", repr(os.environ.get("AZURE_OPENAI_DEPLOYMENT")))
print("  OPENAI model (deployment name) =", repr(OPENAI_MODEL_ID))
print("  Number of available blob_names sample pool:", len(blob_names) if 'blob_names' in globals() else "MISSING")

# Quick sanity: make sure blob_names exists
if 'blob_names' not in globals():
    raise RuntimeError("ERROR: 'blob_names' is not defined in this notebook. Ensure you populate blob_names before running this cell.")

# Quick test download to fail fast (uses your download_blob_to_pil)
test_blob = blob_names[0] if len(blob_names) > 0 else None
if test_blob:
    test_blob_full = test_blob
    print("\nRunning single test download for:", test_blob_full)
    try:
        tp = download_blob_to_pil(test_blob_full)
        print("  Test download OK — image size:", tp.size)
    except Exception as e:
        print("  Test download FAILED -- exception type:", type(e).__name__)
        print("  Exception message:", str(e))
        print("  Common causes: wrong AZURE_BLOB_CONTAINER env var, wrong IMAGE_PREFIX, wrong AZURE_CONN_STR, or blob path doesn't exist.")
        print("  Continuing to main loop so each pair will show per-row errors.")
else:
    print("No blobs found in blob_names to test; aborting.")
    raise RuntimeError("No blob names available for sampling.")

# Get pairs from your random sampler
print("\nSelecting random image pairs...")
source_rows = get_source_votes(conn, N)
print(f"Selected {len(source_rows)} pairs to process.\n")

pbar = tqdm(source_rows, desc="Processing Pairs")

for row in pbar:
    clean_name1 = row['image_name1']
    clean_name2 = row['image_name2']

    blob_name1 = clean_name1
    blob_name2 = clean_name2


    # AI Inference: call your existing model helper (unchanged)
    try:
        raw_txt = call_model_with_public_urls(blob_name1, blob_name2, FULL_PROMPT_TEXT)
    except Exception as e:
        print(f"[Model call] Exception for pair {clean_name1}/{clean_name2}: {type(e).__name__}: {e}")
        # do NOT crash the whole run — skip this pair
        pbar.set_postfix(status="model_err")
        continue

    # If call returned something unexpected, print it (helpful for debugging)
    if raw_txt is None:
        print(f"[Model call] Empty response for {clean_name1}/{clean_name2} — skipping")
        pbar.set_postfix(status="model_empty")
        continue

    # Parse Result (same parser you used)
    match = re.search(r"^(?:\[VOTE\]\s*[:\-]?\s*)?\s*([0-2])\s*[:\-]\s*(.*)$",
                      str(raw_txt).strip(), re.MULTILINE | re.IGNORECASE)

    if not match:
        print(f"\nParse error for {clean_name1}/{clean_name2}. Raw content: '{raw_txt}'")
        pbar.set_postfix(status="parse_err")
        continue

    ai_vote_val = int(match.group(1))
    reason_text = match.group(2).strip()

    rec = {
        "image_name1": clean_name1,
        "image_name2": clean_name2,
        "vote": ai_vote_val,
        "model_id": OPENAI_MODEL_ID,
        "user_id": WORKER_ID,
        "reason": reason_text,
    }

    # Insert using your existing DB helper
    try:
        upsert_vote(conn, rec)
        insert_count += 1
    except Exception as e:
        print("DB insert error:", type(e).__name__, e)
        conn.rollback()
        continue

    results.append({"image1": clean_name1, "image2": clean_name2, "ai_vote": ai_vote_val, "reason": reason_text})

    # Optional display (keeps same signature)
    try:
        display_pair_url(clean_name1, clean_name2, ai_vote_val, None, reason_text, clean_name1, clean_name2)
    except Exception:
        pass

    # Batch commit
    if insert_count % BATCH_COMMIT == 0:
        try:
            conn.commit()
        except Exception:
            conn.rollback()

    pbar.set_postfix(inserted=insert_count)
    time.sleep(RATE_LIMIT_SECONDS)

# Final commit & cleanup
try:
    conn.commit()
except Exception:
    conn.rollback()
conn.close()

# Final Stats & display
print("\n" + "="*40)
print("PROCESSING COMPLETE")
print("="*40)
print(f"Total Pairs Attempted: {len(source_rows)}")
print(f"Records inserted:      {insert_count}")
if results:
    df = pd.DataFrame(results)
    cols = ["image1", "image2", "ai_vote", "reason"]
    display(df[cols].head(50))
else:
    print("No results generated.")


In [ ]:
# GPT 5
# Colab Cell 4 (Modified for GPT-4o + Database Source + Comparison)
import os, io, json, time, base64, hashlib, random, re
from tqdm import tqdm
from PIL import Image
import requests
import mysql.connector
from datetime import datetime
from time import sleep
import uuid
from IPython.display import display, HTML
import pandas as pd
from openai import AzureOpenAI
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ---------- CONFIG ----------
# ---------- AZURE OPENAI CONFIG (minimal changes) ----------

OPENAI_MODEL_ID = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5-chat")
AZURE_OPENAI_ENDPOINT = os.getenv("DEPLOYMENT_NAME","https://codri-mll1iu1g-swedencentral.services.ai.azure.com")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "YOUR_AZURE_OPENAI_API_KEY")

# Construct the client for Azure
client = AzureOpenAI(
    api_version=os.getenv("OPENAI_API_VERSION", "2024-12-01-preview"),
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
)


WORKER_ID = os.getenv("WORKER_ID", str(uuid.uuid4()))

N=10000
BATCH_COMMIT = 1
RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.1"))
RETRY_MAX = 3
IMAGE_PREFIX = "training/training/Images/" # Prefix to add to DB filenames

FULL_PROMPT_TEXT = (
    "You will see two images: Image A (first) and Image B (second).\n\n"
    "TASK: Decide which image looks SAFER for cyclists.\n\n"
    "Respond with ONE line only in the exact required format:\n"
    "[VOTE]: [Vote Number] - [reason]\n\n"
    "Where Vote Number is:\n"
    "  0 - Both look equally safe\n"
    "  1 - Image A looks safer\n"
    "  2 - Image B looks safer\n\n"
    "Keep your reason concise. DO NOT include any extra text.\n\n"
    "Examples (Use this format EXACTLY):\n"
    "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
    "[VOTE]: 2 - Image B has fewer vehicles present.\n"
    "[VOTE]: 0 - Both show similar traffic conditions.\n"
    "\nNow decide which image looks safer."
)

# Stats tracking
results = []
agreement_stats = {"matches": 0, "total": 0}

def pil_to_data_uri(pil_img, quality=85):
    b64 = pil_to_base64_jpeg(pil_img, quality=quality)
    return f"data:image/jpeg;base64,{b64}"

def display_pair_url(image_1, image_2, ai_vote, human_vote, reason, name1, name2, max_width=320):
    uri1 = "https://cycleai.net/london/"+image_1
    uri2 = "https://cycleai.net/london/"+image_2

    # Color code match status
    match_color = "green"

    caption = (
        f"<b>AI Vote:</b> {ai_vote}<br>"
        f"<b>Reason:</b> {reason}<br>"
        f"<b>Files:</b> {name1} — {name2}"
    )

    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>A: {name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>B: {name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

def display_pair(pil1, pil2, ai_vote, human_vote, reason, name1, name2, max_width=320):
    uri1 = pil_to_data_uri(pil1)
    uri2 = pil_to_data_uri(pil2)

    # Color code match status
    match_color = "green"

    caption = (
        f"<b>AI Vote:</b> {ai_vote}<br>"
        f"<b>Reason:</b> {reason}<br>"
        f"<b>Files:</b> {name1} — {name2}"
    )

    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>A: {name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>B: {name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

# ---------- UTILITIES ----------
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format="JPEG", quality=quality)
    return base64.b64encode(buff.getvalue()).decode("utf-8")

def idempotency_key(name1, name2):
    s = f"{name1}|{name2}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def upsert_vote(conn, rec):
    cur = conn.cursor()
    sql = """
    INSERT INTO perception_poll_votes_ai
      (image_name1, image_name2, vote, model_id, user_id, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec.get("reason"))
    cur.execute(sql, params)
    cur.close()

def get_source_votes(conn, limit):
    rows = []
    for _ in range(limit):
        img1, img2 = random.sample(blob_names, 2)
        row = {
            "image_name1": img1,
            "image_name2": img2,
        }

        rows.append(row)

    return rows

def call_model_with_public_urls(image_name1, image_name2, prompt_text, image_angle1="", image_angle2=""):
    try:
        uri1 = "https://cycleai.net/london/"+image_name1
        uri2 = "https://cycleai.net/london/"+image_name2

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_completion_tokens=1064
        )

        return response.choices[0].message.content
    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise




def call_model_openai(prompt_text, pil1, pil2):
    try:
        uri1 = pil_to_data_uri(pil1)
        uri2 = pil_to_data_uri(pil2)

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_completion_tokens=1064
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise

print(f"Starting AI-only inference loop. Target N={N}, Model={OPENAI_MODEL_ID}")
conn = get_mysql_conn()
insert_count = 0

# SHOW ENV + basic config that commonly causes mistakes
print("CONFIG CHECK:")
print("  IMAGE_PREFIX =", repr(IMAGE_PREFIX))
print("  AZURE_BLOB_CONTAINER =", repr(os.environ.get("AZURE_BLOB_CONTAINER")))
print("  AZURE_CONN_STR =", "SET" if os.environ.get("AZURE_CONN_STR") else "MISSING")
print("  AZURE_OPENAI_ENDPOINT =", repr(os.environ.get("AZURE_OPENAI_ENDPOINT")))
print("  AZURE_OPENAI_DEPLOYMENT =", repr(os.environ.get("AZURE_OPENAI_DEPLOYMENT")))
print("  OPENAI model (deployment name) =", repr(OPENAI_MODEL_ID))
print("  Number of available blob_names sample pool:", len(blob_names) if 'blob_names' in globals() else "MISSING")

# Quick sanity: make sure blob_names exists
if 'blob_names' not in globals():
    raise RuntimeError("ERROR: 'blob_names' is not defined in this notebook. Ensure you populate blob_names before running this cell.")

# Quick test download to fail fast (uses your download_blob_to_pil)
test_blob = blob_names[0] if len(blob_names) > 0 else None
if test_blob:
    test_blob_full = test_blob
    print("\nRunning single test download for:", test_blob_full)
    try:
        tp = download_blob_to_pil(test_blob_full)
        print("  Test download OK — image size:", tp.size)
    except Exception as e:
        print("  Test download FAILED -- exception type:", type(e).__name__)
        print("  Exception message:", str(e))
        print("  Common causes: wrong AZURE_BLOB_CONTAINER env var, wrong IMAGE_PREFIX, wrong AZURE_CONN_STR, or blob path doesn't exist.")
        print("  Continuing to main loop so each pair will show per-row errors.")
else:
    print("No blobs found in blob_names to test; aborting.")
    raise RuntimeError("No blob names available for sampling.")

# Get pairs from your random sampler
print("\nSelecting random image pairs...")
source_rows = get_source_votes(conn, N)
print(f"Selected {len(source_rows)} pairs to process.\n")

pbar = tqdm(source_rows, desc="Processing Pairs")

for row in pbar:
    clean_name1 = row['image_name1']
    clean_name2 = row['image_name2']

    blob_name1 = clean_name1
    blob_name2 = clean_name2


    # AI Inference: call your existing model helper (unchanged)
    try:
        raw_txt = call_model_with_public_urls(blob_name1, blob_name2, FULL_PROMPT_TEXT)
    except Exception as e:
        print(f"[Model call] Exception for pair {clean_name1}/{clean_name2}: {type(e).__name__}: {e}")
        # do NOT crash the whole run — skip this pair
        pbar.set_postfix(status="model_err")
        continue

    # If call returned something unexpected, print it (helpful for debugging)
    if raw_txt is None:
        print(f"[Model call] Empty response for {clean_name1}/{clean_name2} — skipping")
        pbar.set_postfix(status="model_empty")
        continue

    # Parse Result (same parser you used)
    match = re.search(r"^(?:\[VOTE\]\s*[:\-]?\s*)?\s*([0-2])\s*[:\-]\s*(.*)$",
                      str(raw_txt).strip(), re.MULTILINE | re.IGNORECASE)

    if not match:
        print(f"\nParse error for {clean_name1}/{clean_name2}. Raw content: '{raw_txt}'")
        pbar.set_postfix(status="parse_err")
        continue

    ai_vote_val = int(match.group(1))
    reason_text = match.group(2).strip()

    rec = {
        "image_name1": clean_name1,
        "image_name2": clean_name2,
        "vote": ai_vote_val,
        "model_id": OPENAI_MODEL_ID,
        "user_id": WORKER_ID,
        "reason": reason_text,
    }

    # Insert using your existing DB helper
    try:
        upsert_vote(conn, rec)
        insert_count += 1
    except Exception as e:
        print("DB insert error:", type(e).__name__, e)
        conn.rollback()
        continue

    results.append({"image1": clean_name1, "image2": clean_name2, "ai_vote": ai_vote_val, "reason": reason_text})

    # Optional display (keeps same signature)
    try:
        display_pair_url(clean_name1, clean_name2, ai_vote_val, None, reason_text, clean_name1, clean_name2)
    except Exception:
        pass

    # Batch commit
    if insert_count % BATCH_COMMIT == 0:
        try:
            conn.commit()
        except Exception:
            conn.rollback()

    pbar.set_postfix(inserted=insert_count)
    time.sleep(RATE_LIMIT_SECONDS)

# Final commit & cleanup
try:
    conn.commit()
except Exception:
    conn.rollback()
conn.close()

# Final Stats & display
print("\n" + "="*40)
print("PROCESSING COMPLETE")
print("="*40)
print(f"Total Pairs Attempted: {len(source_rows)}")
print(f"Records inserted:      {insert_count}")
if results:
    df = pd.DataFrame(results)
    cols = ["image1", "image2", "ai_vote", "reason"]
    display(df[cols].head(50))
else:
    print("No results generated.")


In [ ]:
# gpt 5-2

# GPT 5
# Colab Cell 4 (Modified for GPT-4o + Database Source + Comparison)
import os, io, json, time, base64, hashlib, random, re
from tqdm import tqdm
from PIL import Image
import requests
import mysql.connector
from datetime import datetime
from time import sleep
import uuid
from IPython.display import display, HTML
import pandas as pd
from openai import AzureOpenAI
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ---------- CONFIG ----------
# ---------- AZURE OPENAI CONFIG (minimal changes) ----------

OPENAI_MODEL_ID = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.2")
AZURE_OPENAI_ENDPOINT = os.getenv("ENDPOINT_NAME","https://YOUR_AZURE_OPENAI_RESOURCE.openai.azure.com/")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "YOUR_AZURE_OPENAI_API_KEY")

# Construct the client for Azure
client = AzureOpenAI(
    api_version=os.getenv("OPENAI_API_VERSION", "2024-12-01-preview"),
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
)


WORKER_ID = os.getenv("WORKER_ID", str(uuid.uuid4()))

N=50000
BATCH_COMMIT = 1
RATE_LIMIT_SECONDS = float(os.getenv("RATE_LIMIT_SECONDS", "0.1"))
RETRY_MAX = 3
IMAGE_PREFIX = "training/training/Images/" # Prefix to add to DB filenames

FULL_PROMPT_TEXT = (
    "You will see two images: Image A (first) and Image B (second).\n\n"
    "TASK: Decide which image looks SAFER for cyclists.\n\n"
    "Respond with ONE line only in the exact required format:\n"
    "[VOTE]: [Vote Number] - [reason]\n\n"
    "Where Vote Number is:\n"
    "  0 - Both look equally safe\n"
    "  1 - Image A looks safer\n"
    "  2 - Image B looks safer\n\n"
    "Keep your reason concise. DO NOT include any extra text.\n\n"
    "Examples (Use this format EXACTLY):\n"
    "[VOTE]: 1 - Image A has a dedicated bike lane.\n"
    "[VOTE]: 2 - Image B has fewer vehicles present.\n"
    "[VOTE]: 0 - Both show similar traffic conditions.\n"
    "\nNow decide which image looks safer."
)

# Stats tracking
results = []
agreement_stats = {"matches": 0, "total": 0}

def pil_to_data_uri(pil_img, quality=85):
    b64 = pil_to_base64_jpeg(pil_img, quality=quality)
    return f"data:image/jpeg;base64,{b64}"

def display_pair_url(image_1, image_2, ai_vote, human_vote, reason, name1, name2, max_width=320):
    uri1 = "https://cycleai.net/london/"+image_1
    uri2 = "https://cycleai.net/london/"+image_2

    # Color code match status
    match_color = "green"

    caption = (
        f"<b>AI Vote:</b> {ai_vote}<br>"
        f"<b>Reason:</b> {reason}<br>"
        f"<b>Files:</b> {name1} — {name2}"
    )

    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>A: {name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>B: {name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

def display_pair(pil1, pil2, ai_vote, human_vote, reason, name1, name2, max_width=320):
    uri1 = pil_to_data_uri(pil1)
    uri2 = pil_to_data_uri(pil2)

    # Color code match status
    match_color = "green"

    caption = (
        f"<b>AI Vote:</b> {ai_vote}<br>"
        f"<b>Reason:</b> {reason}<br>"
        f"<b>Files:</b> {name1} — {name2}"
    )

    html = f"""
    <div style="display:flex; align-items:flex-start; gap:12px; margin-bottom:10px;">
      <div style="text-align:center;">
        <img src="{uri1}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>A: {name1}</small>
      </div>
      <div style="text-align:center;">
        <img src="{uri2}" style="max-width:{max_width}px; height:auto; display:block; border:1px solid #ccc;" /><br/>
        <small>B: {name2}</small>
      </div>
      <div style="flex:1; padding-left:8px;">
        {caption}
      </div>
    </div>
    """
    display(HTML(html))

# ---------- UTILITIES ----------
def pil_to_base64_jpeg(pil_img, quality=85):
    buff = io.BytesIO()
    pil_img.save(buff, format="JPEG", quality=quality)
    return base64.b64encode(buff.getvalue()).decode("utf-8")

def idempotency_key(name1, name2):
    s = f"{name1}|{name2}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def upsert_vote(conn, rec):
    cur = conn.cursor()
    sql = """
    INSERT INTO perception_poll_votes_ai
      (image_name1, image_name2, vote, model_id, user_id, reason)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    params = (rec["image_name1"], rec["image_name2"], rec["vote"], rec["model_id"], rec["user_id"], rec.get("reason"))
    cur.execute(sql, params)
    cur.close()

def get_source_votes(conn, limit):
    rows = []
    for _ in range(limit):
        img1, img2 = random.sample(blob_names, 2)
        row = {
            "image_name1": img1,
            "image_name2": img2,
        }

        rows.append(row)

    return rows

def call_model_with_public_urls(image_name1, image_name2, prompt_text, image_angle1="", image_angle2=""):
    try:
        uri1 = "https://cycleai.net/london/"+image_name1
        uri2 = "https://cycleai.net/london/"+image_name2

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_completion_tokens=1064
        )

        return response.choices[0].message.content
    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise




def call_model_openai(prompt_text, pil1, pil2):
    try:
        uri1 = pil_to_data_uri(pil1)
        uri2 = pil_to_data_uri(pil2)

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Image A:"},
                    {"type": "image_url", "image_url": {"url": uri1}},
                    {"type": "text", "text": "\nImage B:"},
                    {"type": "image_url", "image_url": {"url": uri2}},
                    {"type": "text", "text": f"\n\n{prompt_text}"}
                ]
            }
        ]

        response = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=messages,
            max_completion_tokens=1064
        )

        return response.choices[0].message.content

    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        raise

print(f"Starting AI-only inference loop. Target N={N}, Model={OPENAI_MODEL_ID}")
conn = get_mysql_conn()
insert_count = 0

# SHOW ENV + basic config that commonly causes mistakes
print("CONFIG CHECK:")
print("  IMAGE_PREFIX =", repr(IMAGE_PREFIX))
print("  AZURE_BLOB_CONTAINER =", repr(os.environ.get("AZURE_BLOB_CONTAINER")))
print("  AZURE_CONN_STR =", "SET" if os.environ.get("AZURE_CONN_STR") else "MISSING")
print("  AZURE_OPENAI_ENDPOINT =", repr(os.environ.get("AZURE_OPENAI_ENDPOINT")))
print("  AZURE_OPENAI_DEPLOYMENT =", repr(os.environ.get("AZURE_OPENAI_DEPLOYMENT")))
print("  OPENAI model (deployment name) =", repr(OPENAI_MODEL_ID))
print("  Number of available blob_names sample pool:", len(blob_names) if 'blob_names' in globals() else "MISSING")

# Quick sanity: make sure blob_names exists
if 'blob_names' not in globals():
    raise RuntimeError("ERROR: 'blob_names' is not defined in this notebook. Ensure you populate blob_names before running this cell.")

# Quick test download to fail fast (uses your download_blob_to_pil)
test_blob = blob_names[0] if len(blob_names) > 0 else None
if test_blob:
    test_blob_full = test_blob
    print("\nRunning single test download for:", test_blob_full)
    try:
        tp = download_blob_to_pil(test_blob_full)
        print("  Test download OK — image size:", tp.size)
    except Exception as e:
        print("  Test download FAILED -- exception type:", type(e).__name__)
        print("  Exception message:", str(e))
        print("  Common causes: wrong AZURE_BLOB_CONTAINER env var, wrong IMAGE_PREFIX, wrong AZURE_CONN_STR, or blob path doesn't exist.")
        print("  Continuing to main loop so each pair will show per-row errors.")
else:
    print("No blobs found in blob_names to test; aborting.")
    raise RuntimeError("No blob names available for sampling.")

# Get pairs from your random sampler
print("\nSelecting random image pairs...")
source_rows = get_source_votes(conn, N)
print(f"Selected {len(source_rows)} pairs to process.\n")

pbar = tqdm(source_rows, desc="Processing Pairs")

for row in pbar:
    clean_name1 = row['image_name1']
    clean_name2 = row['image_name2']

    blob_name1 = clean_name1
    blob_name2 = clean_name2


    # AI Inference: call your existing model helper (unchanged)
    try:
        raw_txt = call_model_with_public_urls(blob_name1, blob_name2, FULL_PROMPT_TEXT)
    except Exception as e:
        print(f"[Model call] Exception for pair {clean_name1}/{clean_name2}: {type(e).__name__}: {e}")
        # do NOT crash the whole run — skip this pair
        pbar.set_postfix(status="model_err")
        continue

    # If call returned something unexpected, print it (helpful for debugging)
    if raw_txt is None:
        print(f"[Model call] Empty response for {clean_name1}/{clean_name2} — skipping")
        pbar.set_postfix(status="model_empty")
        continue

    # Parse Result (same parser you used)
    match = re.search(r"^(?:\[VOTE\]\s*[:\-]?\s*)?\s*([0-2])\s*[:\-]\s*(.*)$",
                      str(raw_txt).strip(), re.MULTILINE | re.IGNORECASE)

    if not match:
        print(f"\nParse error for {clean_name1}/{clean_name2}. Raw content: '{raw_txt}'")
        pbar.set_postfix(status="parse_err")
        continue

    ai_vote_val = int(match.group(1))
    reason_text = match.group(2).strip()

    rec = {
        "image_name1": clean_name1,
        "image_name2": clean_name2,
        "vote": ai_vote_val,
        "model_id": OPENAI_MODEL_ID,
        "user_id": WORKER_ID,
        "reason": reason_text,
    }

    # Insert using your existing DB helper
    try:
        upsert_vote(conn, rec)
        insert_count += 1
    except Exception as e:
        print("DB insert error:", type(e).__name__, e)
        conn.rollback()
        continue

    results.append({"image1": clean_name1, "image2": clean_name2, "ai_vote": ai_vote_val, "reason": reason_text})

    # Optional display (keeps same signature)
    try:
        display_pair_url(clean_name1, clean_name2, ai_vote_val, None, reason_text, clean_name1, clean_name2)
    except Exception:
        pass

    # Batch commit
    if insert_count % BATCH_COMMIT == 0:
        try:
            conn.commit()
        except Exception:
            conn.rollback()

    pbar.set_postfix(inserted=insert_count)
    time.sleep(RATE_LIMIT_SECONDS)

# Final commit & cleanup
try:
    conn.commit()
except Exception:
    conn.rollback()
conn.close()

# Final Stats & display
print("\n" + "="*40)
print("PROCESSING COMPLETE")
print("="*40)
print(f"Total Pairs Attempted: {len(source_rows)}")
print(f"Records inserted:      {insert_count}")
if results:
    df = pd.DataFrame(results)
    cols = ["image1", "image2", "ai_vote", "reason"]
    display(df[cols].head(50))
else:
    print("No results generated.")
